# Libs

In [ ]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.manifold import TSNE, MDS

# 1. Textual data preprocessing

In [ ]:
## read and view dataset
df_dataset = pd.read_csv('steam_games.csv', decimal='.')
df_dataset

- Como os jogos podem pertencer a mais de um gênero e nesses casos estão separados por ";" converti para listas com o objetivo de facilitar a manipulação

In [ ]:
## remove unnecessary columns
df_dataset.drop(columns=['appid', 'release_date', 'positive_ratings', 'negative_ratings', 'average_playtime', 'price'], inplace=True, errors='ignore')
## convert genres column to list
df_dataset['genres'] = df_dataset['genres'].str.split(';')
df_dataset

- O título do jogo carrega palavras-chave que podem ajudar a definir o gênero do jogo, por isso optei por incluir também o título na análise. 
- Percebi que a descrição curta já carrega muitas informações relevantes e assim inialmente optei por utilizar ela com objetivo de reduzir ruído para o TF-IDF.

In [ ]:
## remove stopwords and lemmatize text
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def clean_text(text):
    # remove especial characters and lower case
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    # tokenization, stopwords removal, and lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

# Creating unified corpus for text features title + description
df_dataset['text_feature'] = df_dataset['name'] + " " + df_dataset['short_description']
df_dataset['clean_text'] = df_dataset['text_feature'].apply(clean_text)
df_dataset

# 2. TF-IDF Matrix

In [ ]:
tfidf = TfidfVectorizer(encoding='utf-8-sig', min_df=5)
X_tfidf = tfidf.fit_transform(df_dataset['clean_text'])
X_tfidf

# 3. PCA to Dimmensionally Reduction

In [ ]:
## PCA dont use with sparse matrix, so i use TruncatedSVD to reduce dimensions
# pca = PCA(n_components=0.95, random_state=42)
# X_pca = pca.fit_transform(X_tfidf.toarray())
# X_pca

In [ ]:
svd = TruncatedSVD(n_components=5000, random_state=42) 
X_svd = svd.fit_transform(X_tfidf) 
print(f"Variância explicada: {svd.explained_variance_ratio_.sum():.2f}")

# 4. Clustering

## Visualize data in lower dimension

### t-SNE

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=50,
    early_exaggeration=12.0,
    learning_rate='auto',
    max_iter=1000,
    n_iter_without_progress=500,
    min_grad_norm=1e-7,
    metric='euclidean',
    init='pca',
    random_state=42,
    method='barnes_hut',
    angle=0.5,
    verbose=0,
    n_jobs=None
)

X_embedded = tsne.fit_transform(X_svd)
plt.figure(figsize=(8, 6))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], s=40, alpha=0.7, c='red')
plt.title("t-SNE visualization of SVD-reduced data")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

### MDS

In [ ]:
mds = MDS(n_components=2, n_init=5, random_state=42)
X_mds = mds.fit_transform(X_svd)
print(X_mds.shape)

plt.figure(figsize=(8, 6))
plt.scatter(X_mds[:, 0], X_mds[:, 1], s=40, alpha=0.7, c='red')
plt.title("Visualização com MDS")
plt.xlabel("Componente 1 (MDS)")
plt.ylabel("Componente 2 (MDS)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## KMeans

In [ ]:
# Definindo K arbitrário (ideal usar método do cotovelo antes)
K = 5 

# Algoritmo A: K-Means
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_svd)

## Hierarchical Clustering

## Spectral Clustering